# LAMMPS.js notebooks

Welcome! These notebooks run [LAMMPS](https://www.lammps.org) — the classical
molecular dynamics code — **entirely in your browser**. There is no server:
LAMMPS is compiled to WebAssembly ([lammps.js](https://github.com/lammps/lammps.js))
and driven from JavaScript cells through a
[JupyterLite](https://jupyterlite.readthedocs.io/) JavaScript kernel.

- Run a cell with **Shift+Enter**.
- Everything you edit is saved in your browser's local storage. To restore a
  tutorial to its original state: right-click it in the file browser and pick
  *Revert to original*  (or delete it — the built-in copy reappears).
- The first simulation cell downloads the LAMMPS wasm module (~10 MB) — it is
  cached after that.

Also see the [interactive API docs](../docs/) and the [playground](../) where
you can paste any LAMMPS script.

In [ ]:
// Load lammps.js (served by this site under ./lammps/). Run this cell first.
// The site root is derived from wherever this code runs: the kernel iframe
// inherits the page URL ({site}/lab/…), the worker kernel lives under
// {site}/extensions/….
const base = globalThis.document?.baseURI ?? location.href;
globalThis.SITE ??= base.replace(/(extensions|lab|notebooks|files|tree|repl|consoles|edit)\/.*$/, "");
globalThis.LammpsClient ??= (await import(new URL("lammps/client.js", globalThis.SITE))).LammpsClient;
"lammps.js loaded ✓"

## Your first simulation

A Lennard-Jones fcc crystal, melted with an NVE run. LAMMPS's log output goes
to `console.log`, so it appears right below the cell — exactly what you would
see in a terminal.

In [ ]:
const lammps = await LammpsClient.create({ print: (line) => console.log(line) });
lammps.start();

lammps.runScript(`
  units         lj
  atom_style    atomic
  lattice       fcc 0.8442
  region        box block 0 4 0 4 0 4
  create_box    1 box
  create_atoms  1 box
  mass          1 1.0
  velocity      all create 3.0 87287
  pair_style    lj/cut 2.5
  pair_coeff    1 1 1.0 1.0 2.5
  fix           1 all nve
  thermo        200
  run           1000
`);

console.log("");
console.log("atoms simulated:", lammps.syncParticles().count);
lammps.dispose();

That's a real MD run: 256 atoms, 1000 timesteps, computed by the same C++
LAMMPS code you would run on a cluster — just compiled to WebAssembly.

## Where to go next

| Notebook | What it covers |
|---|---|
| [basics/01-getting-started](basics/01-getting-started.ipynb) | The `LammpsClient` API: creating a session, running scripts and commands |
| [basics/02-scripts-and-files](basics/02-scripts-and-files.ipynb) | The in-memory filesystem: data files, dumps, loading scripts from the site |
| [basics/03-live-data](basics/03-live-data.ipynb) | Watching a run live: step callbacks and compute scalars |
| [basics/04-snapshots-and-analysis](basics/04-snapshots-and-analysis.ipynb) | Positions as typed arrays; computing mean-squared displacement in JS |

More series (materials science with EAM potentials, soft matter, and atomify's
example library) are planned — see
[NOTEBOOK_TUTORIALS.md](https://github.com/lammps/lammps.js/blob/master/NOTEBOOK_TUTORIALS.md).